<a href="https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
My lane (Lane 2: Refresh / Content Opportunity Scoring) is a **ranking / scoring** problem, built on top of a classification foundation.

The decision it supports is "which pages should a content manager look at first?" — that's a "which ones first?" question, not a "what group is this?" question. So the final output has to be an ordered priority list, not just a label.

But under the hood, the score I rank by comes from a classifier: I'll estimate the *probability* that a page is declining, and use that probability as the priority score. Higher predicted probability of decline = higher up the queue. So the task type is scoring/ranking, and the model that produces the score is a binary classifier.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
The target is `is_declining_label`, a binary column: 1 if a page's `trend_direction` is `"down"`, 0 otherwise.

`trend_direction` itself is computed from `trend_pct`, which compares `impressions_last_30d` against `impressions_prev_30d` (a page is "down" if impressions fell more than 20%). So the label is **observed** — it comes from real, measured impression counts over two actual time windows, not from someone's opinion or a manually-assigned tag.

One important rule this creates: since `trend_direction` and `trend_pct` are literally used to build the label, they can **never** be features. Using them would just teach the model to reverse-engineer its own label (leakage), not to find any real predictive signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
**Precision@K.** Since the real action is a content manager working down a ranked list starting from the top, what matters is: of the top K pages my model flags, how many are actually declining?

The number to beat is the base rate: **54.2%** of all pages in the starter dataset are declining (16,262 of 30,000). That's unusually high — it means a fixed rule like "assume everything is declining" would already look "correct" over half the time. So my model only proves its worth if precision@K for the top-ranked pages is *meaningfully above* 54.2%, not just above 50%. If it can't beat that number, it isn't earning its place over a coin flip.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
One row = one content page (`content_id`), pseudonymized and belonging to one of 32 clients. Below I load the starter CSV, derive the label myself (since it isn't shipped pre-computed), and show what the target column looks like.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    # Colab fallback: clone the repo, then read from it
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

print("Rows, columns:", df.shape)
print("One row =", df["content_id"].nunique(), "unique content pages")

# Derive the label myself — it's not shipped pre-computed in the raw 44-column CSV
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("\nLabel distribution:")
print(df["is_declining_label"].value_counts())
print("\nBase rate (share declining):", round(df["is_declining_label"].mean() * 100, 1), "%")

df[["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]].head(10)

Cloning into 'ML-internship'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 13), reused 62 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.84 MiB | 15.84 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Rows, columns: (30000, 44)
One row = 30000 unique content pages

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (share declining): 54.2 %


,content_id,client_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,down,-38.9,1
6,content_9a34b442b552,client_8722616204,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,down,-29.2,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
If a fixed rule worked, I'd expect declining and non-declining pages to sit in clearly separate ranges on the signals that should matter — CTR, engagement rate, freshness. Below I check that directly by grouping on the label.

Running that check: CTR *means* do differ some between groups (0.73% vs 0.32%), but the 25th/50th/75th percentiles for declining and non-declining pages land almost on top of each other — the median CTR is 0.04% either way. Same story for `engagement_rate`, `days_since_last_update`, and `ai_traffic_pct`: the middle of each distribution overlaps heavily across the two groups. There's no clean cutoff — no single "if CTR < X, flag as declining" rule that wouldn't misclassify huge numbers of pages on both sides. The signal is real (means shift) but tangled across many overlapping variables at once, which is exactly the case where a rule breaks down and a model that can weigh several signals together earns its place.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["ctr", "engagement_rate", "days_since_last_update", "ai_traffic_pct"]:
    summary = df.groupby("is_declining_label")[col].describe()[["mean", "25%", "50%", "75%"]]
    print(f"--- {col} ---")
    print(summary)
    print()

--- ctr ---
                        mean  25%   50%   75%
is_declining_label                           
0                   0.731611  0.0  0.04  0.30
1                   0.324138  0.0  0.08  0.27

--- engagement_rate ---
                        mean  25%  50%     75%
is_declining_label                            
0                   2.649728  0.0  0.0  1.4800
1                   2.437193  0.0  0.0  1.2475

--- days_since_last_update ---
                         mean   25%   50%    75%
is_declining_label                              
0                   42.372543  20.0  20.0  104.0
1                   49.245788  20.0  20.0  104.0

--- ai_traffic_pct ---
                        mean  25%  50%  75%
is_declining_label                         
0                   0.748514  0.0  0.0  0.0
1                   0.784824  0.0  0.0  0.0



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.